# Heatmap: Traffic

This notebook builds a heatmap of where vehicles are on a busy road. Every frame, each detected vehicle adds a bit of "heat" at its position; over the video the heat builds up where vehicles spend the most time.

1. Detect vehicles with [RF-DETR](https://github.com/roboflow/rf-detr) using tiled inference.
2. Accumulate their positions with Supervision's `sv.HeatMapAnnotator`.
3. Look at the final heatmap on its own and discuss what it actually measures.

## Setup

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
from rfdetr import RFDETRNano
from rfdetr.assets.coco_classes import COCO_CLASSES
from supervision.assets import VideoAssets, download_assets
from tqdm.auto import tqdm

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## Download the video

`VEHICLES_2` shows a sunken multi-lane road from a fixed camera: a busy road below and a cross street with a traffic light at the top. As with zones, a heatmap only makes sense with a fixed camera: if the camera moved, the same pixel would point at a different spot on the road.

In [ ]:
SOURCE_VIDEO_PATH = download_assets(VideoAssets.VEHICLES_2, directory=DATA_DIR)
TARGET_VIDEO_PATH = OUTPUT_DIR / "vehicles_2_heatmap.mp4"
HEATMAP_IMAGE_PATH = OUTPUT_DIR / "vehicles_2_heatmap.png"

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
video_info

In [ ]:
background = next(sv.get_video_frames_generator(SOURCE_VIDEO_PATH))
sv.plot_image(background, size=(12, 7))

## Detect vehicles

Distant cars are small, so we use `sv.InferenceSlicer` again: on this video it finds roughly three times as many vehicles as a single full-frame prediction.

In [ ]:
model = RFDETRNano()

VEHICLE_CLASSES = ["car", "motorcycle", "bus", "truck"]
VEHICLE_CLASS_IDS = [k for k, v in COCO_CLASSES.items() if v in VEHICLE_CLASSES]


def detect_batch(images):
    return model.predict([image[:, :, ::-1].copy() for image in images], threshold=0.5)


slicer = sv.InferenceSlicer(
    callback=detect_batch,
    slice_wh=640,
    overlap_wh=128,
    batch_size=8,
    overlap_metric=sv.OverlapMetric.IOS,
)


def detect_vehicles(frame):
    detections = slicer(frame)
    return detections[np.isin(detections.class_id, VEHICLE_CLASS_IDS)]

In [ ]:
detections = detect_vehicles(background)
print(f"{len(detections)} vehicles detected")

sv.plot_image(sv.BoxAnnotator().annotate(background.copy(), detections), size=(12, 7))

## Build the heatmap

`sv.HeatMapAnnotator` is stateful: it keeps a `heat_mask` array the size of the frame and, on every call, adds a filled circle at each detection's anchor point. The overlay is coloured relative to the hottest pixel so far, from blue (little time) to red (most time).

- **`position`**: which point of the box adds heat. The box centre works well for vehicles.
- **`radius`**: size of each circle; roughly half a car width at this resolution.
- **`kernel_size`**: blur applied to the colours, to smooth the edges.

The annotator draws onto the image it's given, so we always pass a copy of the frame.

We process every second frame (`stride=2`) and write the output at half the frame rate so it still plays in real time. It takes about a minute and a half on an Apple Silicon Mac.

In [ ]:
heatmap_annotator = sv.HeatMapAnnotator(position=sv.Position.CENTER, radius=20, kernel_size=25, opacity=0.5)
box_annotator = sv.BoxAnnotator(thickness=1, color=sv.Color.WHITE)

STRIDE = 2
output_info = sv.VideoInfo(width=video_info.width, height=video_info.height, fps=video_info.fps / STRIDE)
frames = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, stride=STRIDE)

with sv.VideoSink(str(TARGET_VIDEO_PATH), output_info) as sink:
    for frame in tqdm(frames, total=video_info.total_frames // STRIDE + 1):
        detections = detect_vehicles(frame)
        annotated = heatmap_annotator.annotate(frame.copy(), detections)
        annotated = box_annotator.annotate(annotated, detections)
        sink.write_frame(annotated)

## The final heatmap

After the loop, `heatmap_annotator.heat_mask` holds the heat accumulated over the whole video. We can look at it in two ways:

- **Overlay**: the heat drawn on top of an empty frame of the scene (this is what the annotator does).
- **Raw values**: the `heat_mask` array itself, plotted with a colour map. Values span a huge range (a parked-looking car in a jam piles up heat every frame), so a log scale shows the lanes much better than a linear one.

In [ ]:
heat = heatmap_annotator.heat_mask.copy()

overlay = sv.HeatMapAnnotator(position=sv.Position.CENTER, radius=20, kernel_size=25, opacity=0.6)
overlay.heat_mask = heat  # start from the accumulated heat; empty detections add nothing
final_overlay = overlay.annotate(background.copy(), sv.Detections.empty())
cv2.imwrite(str(HEATMAP_IMAGE_PATH), final_overlay)

fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
axes[0].imshow(cv2.cvtColor(final_overlay, cv2.COLOR_BGR2RGB))
axes[0].set_title("Heatmap overlay")
image = axes[1].imshow(np.log1p(heat), cmap="inferno")
axes[1].set_title("log(1 + heat)")
fig.colorbar(image, ax=axes[1], fraction=0.03)
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## What does the heatmap measure?

Heat is added **once per frame per detection**, so it measures *time spent at a pixel*, not *number of vehicles*. Two effects make that visible here:

- **Dwell time.** The hottest band is the cross street at the top, where cars queue at the traffic light and sit in the same spot for many frames.
- **Perspective.** Lanes get hotter towards the far end of the road. Far-away cars look small and cover only a few pixels per frame, so they stay on the same pixels longer than nearby cars moving at the same real speed.

That makes heatmaps great for "where do people or vehicles linger?" (queues, shop windows, bottlenecks), and misleading for "how busy is each lane?" unless you correct for it.

Open `output/vehicles_2_heatmap.mp4` to watch the heat build up over time.

**Things to try**

- Add a tracker (as in the earlier notebooks) and add heat only the first time each `tracker_id` enters a grid cell, to get a traffic-volume map instead of a dwell-time map.
- Build a separate heatmap per class, e.g. trucks only.
- Use `sv.Position.BOTTOM_CENTER` with the people-walking or market-square videos to map where people walk.